**Imports & Constants**

In [121]:
import re
import json
from datetime import datetime, timedelta


**PROMPTS**

In [122]:
PLANNER_PROMPT = """
You are a planning agent.
Given a word problem, produce a concise numbered plan.

Examples:

Q: If a train leaves at 14:30 and arrives at 18:05, how long is the journey?
Plan:
1. Extract times
2. Compute time difference
3. Format duration

Q: Alice has 3 red apples and twice as many green apples.
Plan:
1. Extract base quantity
2. Apply twice-as-many rule
3. Sum totals
"""

EXECUTOR_PROMPT = """
You are an execution agent.
Follow the plan to compute the answer.
Do not expose chain-of-thought.

Examples:
Q: Train 14:30 to 18:05
Answer: 3 hours 35 minutes

Q: 3 red apples, twice green
Answer: 9
"""

VERIFIER_PROMPT = """
You are a verification agent.
Check correctness and consistency.

Examples:
Answer: 3 hours 35 minutes → PASSED
Answer: -5 → FAILED (negative value)
"""


**Planner**

In [123]:
def planner(question: str) -> str:
    return (
        "1. Parse the question\n"
        "2. Extract quantities or time values\n"
        "3. Perform calculations\n"
        "4. Validate constraints\n"
        "5. Format final answer"
    )


**Executo**r

In [124]:
def executor(question: str, plan: str) -> dict:
    q = question.lower()

    # --- Time difference problems ---
    times = re.findall(r'(\d{2}:\d{2})', question)
    if len(times) == 2:
        start = datetime.strptime(times[0], "%H:%M")
        end = datetime.strptime(times[1], "%H:%M")
        if end < start:
            end += timedelta(days=1)

        delta = end - start
        h = delta.seconds // 3600
        m = (delta.seconds % 3600) // 60

        hour_label = "hour" if h == 1 else "hours"
        minute_label = "minute" if m == 1 else "minutes"

        return {
            "answer": f"{h} {hour_label} {m} {minute_label}",
            "explanation": "Computed the difference between the two times.",
            "intermediate": delta
        }

    # --- Twice-as-many arithmetic (apples, balls, etc.) ---
    if "twice as many" in q:
        base = int(re.search(r'(\d+)', question).group(1))
        total = base + (2 * base)

        return {
            "answer": str(total),
            "explanation": "Computed total using the twice-as-many relationship.",
            "intermediate": total
        }

    # --- Meeting slot problems ---
    if "meeting" in q:
        duration = int(re.search(r'(\d+)\s+minutes', question).group(1))
        slots = re.findall(r'(\d{2}:\d{2})–(\d{2}:\d{2})', question)

        valid = []
        for s, e in slots:
            start = datetime.strptime(s, "%H:%M")
            end = datetime.strptime(e, "%H:%M")
            if (end - start).seconds / 60 >= duration:
                valid.append(f"{s}–{e}")

        return {
            "answer": ", ".join(valid),
            "explanation": "Validated each slot against meeting duration.",
            "intermediate": valid
        }

    return {
        "answer": "",
        "explanation": "Unable to solve the question.",
        "intermediate": None
    }


**Verifier**

In [125]:
def verifier(question: str, solution: dict) -> dict:
    if not solution["answer"]:
        return {
            "check_name": "Answer existence",
            "passed": False,
            "details": "No answer produced"
        }

    if solution["answer"].isdigit() and int(solution["answer"]) < 0:
        return {
            "check_name": "Non-negative check",
            "passed": False,
            "details": "Negative value detected"
        }

    if "hour" in solution["answer"]:
        return {
            "check_name": "Time duration check",
            "passed": True,
            "details": "Duration is positive and well formatted"
        }

    return {
        "check_name": "Arithmetic consistency",
        "passed": True,
        "details": "Arithmetic result is valid"
    }


**Agent Loop**

In [126]:
MAX_RETRIES = 2

def solve(question: str) -> dict:
    retries = 0

    while retries <= MAX_RETRIES:
        plan = planner(question)
        solution = executor(question, plan)
        check = verifier(question, solution)

        if check["passed"]:
            return {
                "answer": solution["answer"],
                "status": "success",
                "reasoning_visible_to_user": solution["explanation"],
                "metadata": {
                    "plan": plan,
                    "checks": [check],
                    "retries": retries
                }
            }

        retries += 1

    return {
        "answer": "",
        "status": "failed",
        "reasoning_visible_to_user": "Verification failed after retries.",
        "metadata": {
            "plan": plan,
            "checks": [check],
            "retries": retries
        }
    }


**5 EASY TEST QUESTIONS**

In [127]:
easy_tests = [
    "If a train leaves at 14:30 and arrives at 18:05, how long is the journey?",
    "Alice has 3 red apples and twice as many green apples as red. How many apples does she have in total?",
    "If a class starts at 09:00 and ends at 10:15, how long is the class?",
    "Tom has 4 blue balls and twice as many red balls. How many balls total?",
    "A movie starts at 20:00 and ends at 22:30. How long is the movie?"
]

for q in easy_tests:
    print("\nQUESTION:", q)
    print(json.dumps(solve(q), indent=2))



QUESTION: If a train leaves at 14:30 and arrives at 18:05, how long is the journey?
{
  "answer": "3 hours 35 minutes",
  "status": "success",
  "reasoning_visible_to_user": "Computed the difference between the two times.",
  "metadata": {
    "plan": "1. Parse the question\n2. Extract quantities or time values\n3. Perform calculations\n4. Validate constraints\n5. Format final answer",
    "checks": [
      {
        "check_name": "Time duration check",
        "passed": true,
        "details": "Duration is positive and well formatted"
      }
    ],
    "retries": 0
  }
}

QUESTION: Alice has 3 red apples and twice as many green apples as red. How many apples does she have in total?
{
  "answer": "9",
  "status": "success",
  "reasoning_visible_to_user": "Computed total using the twice-as-many relationship.",
  "metadata": {
    "plan": "1. Parse the question\n2. Extract quantities or time values\n3. Perform calculations\n4. Validate constraints\n5. Format final answer",
    "checks

**3 TRICKY TEST QUESTIONS**

In [128]:
tricky_tests = [
    "A meeting needs 60 minutes. There are free slots: 09:00–09:30, 09:45–10:30, 11:00–12:00. Which slots can fit the meeting?",
    "A train leaves at 23:30 and arrives at 01:00. How long is the journey?",
    "Alice has 0 red apples and twice as many green apples. How many apples total?"
]

for q in tricky_tests:
    print("\nQUESTION:", q)
    print(json.dumps(solve(q), indent=2))



QUESTION: A meeting needs 60 minutes. There are free slots: 09:00–09:30, 09:45–10:30, 11:00–12:00. Which slots can fit the meeting?
{
  "answer": "11:00\u201312:00",
  "status": "success",
  "reasoning_visible_to_user": "Validated each slot against meeting duration.",
  "metadata": {
    "plan": "1. Parse the question\n2. Extract quantities or time values\n3. Perform calculations\n4. Validate constraints\n5. Format final answer",
    "checks": [
      {
        "check_name": "Arithmetic consistency",
        "passed": true,
        "details": "Arithmetic result is valid"
      }
    ],
    "retries": 0
  }
}

QUESTION: A train leaves at 23:30 and arrives at 01:00. How long is the journey?
{
  "answer": "1 hour 30 minutes",
  "status": "success",
  "reasoning_visible_to_user": "Computed the difference between the two times.",
  "metadata": {
    "plan": "1. Parse the question\n2. Extract quantities or time values\n3. Perform calculations\n4. Validate constraints\n5. Format final answer